# 01 — Mapillary API Full Collection | Kota Bandung

Notebook ini menangani **pengumpulan metadata citra Mapillary dari awal sampai dataset kandidat siap dipakai pada tahap download dan screening**.

Fokus notebook:
1. Memuat batas administrasi Kota Bandung.
2. Mempertahankan atau membuat **search grid 1 × 1 km** sebagai unit pembagian area pengumpulan.
3. Mengubah setiap sel menjadi bounding box WGS84.
4. Mengambil metadata citra melalui Mapillary Graph API.
5. Membagi bounding box yang terlalu padat menjadi empat sub-area (*recursive spatial split*).
6. Menyimpan checkpoint per area agar proses dapat dilanjutkan.
7. Menggabungkan hasil, menghapus duplikasi antar-area, memeriksa kecamatan, dan melakukan spatial sampling minimum 10 m dalam sequence yang sama.
8. Mengekspor `candidates_master.csv`, `review_candidates_10m_full.csv`, GeoPackage, dan log pengumpulan.

**Batas pekerjaan notebook ini:** mengumpulkan metadata dan URL thumbnail. Download gambar resolusi yang dipilih, screening kualitas, dan pemilihan gambar untuk PCD dikerjakan pada notebook berikutnya.

> Search grid 1 km bukan analysis grid 250 × 250 m dari tesis ITB. Keduanya mempunyai fungsi berbeda dan tidak boleh digabung.

## Alur kerja

```text
Batas Kota Bandung + batas kecamatan
                │
                ▼
     Search grid 1 × 1 km (±212 sel)
                │
                ▼
     Bounding box WGS84 per sel
                │
                ▼
      Mapillary Graph API /images
                │
       ┌────────┴─────────┐
       │                  │
   Hasil sedikit      Hasil padat / API 500
       │                  │
       │              Bagi bbox menjadi 4
       │              dan query ulang
       └────────┬─────────┘
                ▼
    Checkpoint metadata per search grid
                │
                ▼
  Gabungkan + deduplikasi berdasarkan image_id
                │
                ▼
       Filter titik di Kota Bandung
                │
                ▼
  Spatial sampling 10 m dalam sequence sama
                │
                ▼
       Master + kandidat untuk screening
```

**Cara menjalankan:** jalankan cell secara berurutan. Uji beberapa area terlebih dahulu (`COLLECTION_MODE = "pilot"`). Setelah output pilot diperiksa, ubah ke `all` untuk seluruh search grid.

## 0. Persiapan environment

In [ ]:
# Google Colab
!pip -q install geopandas pyogrio shapely pyproj pandas numpy requests tqdm

In [ ]:
import os
import re
import json
import math
import time
import getpass
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from shapely.geometry import box, Polygon, MultiPolygon
from tqdm.auto import tqdm
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
print("GeoPandas:", gpd.__version__)
print("Pandas:", pd.__version__)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Konfigurasi proyek dan path data

Notebook menggunakan path proyek yang sudah dipakai pada tahap pilot. Pastikan kedua file ini tersedia:

- `01_RAW_DATA/ADMIN/batas_kecamatan_bandung_full.gpkg`
- `03_GIS/processed/bandung_search_grid_1km.gpkg`

Jika search grid belum tersedia, notebook dapat membuat grid 1 km dari batas administrasi.

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/GIS-Trotoar-Bandung")

ADMIN_PATH = PROJECT_ROOT / "01_RAW_DATA/ADMIN/batas_kecamatan_bandung_full.gpkg"
SEARCH_GRID_PATH = PROJECT_ROOT / "03_GIS/processed/bandung_search_grid_1km.gpkg"

MAPILLARY_ROOT = PROJECT_ROOT / "01_RAW_DATA/MAPILLARY"
METADATA_DIR = MAPILLARY_ROOT / "METADATA"
PER_AREA_DIR = METADATA_DIR / "by_grid"
LOG_DIR = METADATA_DIR / "logs"

for path in [METADATA_DIR, PER_AREA_DIR, LOG_DIR, SEARCH_GRID_PATH.parent]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ADMIN_PATH exists:", ADMIN_PATH.exists())
print("SEARCH_GRID_PATH exists:", SEARCH_GRID_PATH.exists())
print("Output metadata:", METADATA_DIR)

## 3. Parameter pengumpulan

Parameter awal mengikuti pendekatan pilot yang sudah digunakan:

- Maksimum `limit` per request: 200.
- Jika satu bbox mengembalikan minimal 100 gambar, bbox akan dibagi menjadi empat.
- Jika API mengembalikan HTTP 500 setelah retry, bbox juga dicoba melalui recursive split.
- Kedalaman maksimum split: 6.
- Ukuran bbox minimum: 0,00015 derajat.
- Jeda antarrekuest: 0,15 detik.

Mulai dari pilot lima area yang pernah digunakan. Ubah `COLLECTION_MODE` menjadi `all` hanya setelah uji pilot berhasil.

In [ ]:
# Pilihan: "pilot", "custom", atau "all"
COLLECTION_MODE = "pilot"
PILOT_AREA_IDS = ["BDG_0103", "BDG_0104", "BDG_0084", "BDG_0085", "BDG_0122"]
CUSTOM_AREA_IDS = []

# Proteksi agar seluruh kota tidak ter-query tanpa sengaja
ALLOW_FULL_COLLECTION = False
RUN_COLLECTION = False          # Ubah menjadi True setelah token dan area diperiksa
RECOLLECT_COMPLETED = False      # False = lanjutkan dari checkpoint yang sudah selesai

# Mapillary API
MAPILLARY_ENDPOINT = "https://graph.mapillary.com/images"
API_FIELDS = [
    "id",
    "computed_geometry",
    "captured_at",
    "thumb_1024_url",
    "thumb_2048_url",
    "sequence",
    "compass_angle",
    "creator",
    "camera_type",
    "width",
    "height",
]
API_LIMIT = 200
SPLIT_THRESHOLD = 100
MAX_SPLIT_DEPTH = 6
MIN_BBOX_DEG = 0.00015
REQUEST_TIMEOUT = 60
MAX_RETRIES = 4
REQUEST_PAUSE = 0.15

# Pengurangan kepadatan foto dalam sequence yang sama
MIN_DISTANCE_METERS = 10.0

# CRS pemrosesan Kota Bandung
PROJECTED_CRS = "EPSG:32748"
GEOGRAPHIC_CRS = "EPSG:4326"

print("Collection mode:", COLLECTION_MODE)
print("Run API collection:", RUN_COLLECTION)
print("API fields:", ", ".join(API_FIELDS))

## 4. Autentikasi Mapillary

Token dimasukkan melalui `getpass` supaya tidak tampil di output notebook. Jangan menulis token langsung di cell, CSV, atau GitHub.

Permintaan Graph API menggunakan header `Authorization: OAuth ...`. Dokumentasi SDK Mapillary merekomendasikan header authorization untuk request Graph API dan tidak menganjurkan pengiriman user token sebagai query parameter. [Dokumentasi autentikasi](https://mapillary.github.io/mapillary-python-sdk/docs/mapillary.models/mapillary.models.client/).

In [ ]:
MAPILLARY_TOKEN = getpass.getpass("Masukkan Mapillary access token: ").strip()
if not MAPILLARY_TOKEN:
    raise ValueError("Token Mapillary belum diisi.")

SESSION = requests.Session()
SESSION.headers.update({
    "Authorization": f"OAuth {MAPILLARY_TOKEN}",
    "User-Agent": "Bandung-Sidewalk-Research/1.0",
})

print("Token diterima. Nilai token tidak ditampilkan.")

## 5. Memuat batas administrasi

Batas kecamatan dipakai untuk membentuk area studi, memeriksa apakah titik citra benar-benar berada di Kota Bandung, dan memberi label kecamatan.

Notebook mengasumsikan GeoPackage berisi poligon kecamatan Kota Bandung dengan field kecamatan seperti `WADMKC`.

In [ ]:
if not ADMIN_PATH.exists():
    raise FileNotFoundError(
        f"Batas administrasi tidak ditemukan: {ADMIN_PATH}. "
        "Simpan batas_kecamatan_bandung_full.gpkg pada folder ADMIN."
    )

admin = gpd.read_file(ADMIN_PATH)
if admin.empty:
    raise ValueError("Layer batas administrasi kosong.")
if admin.crs is None:
    raise ValueError("CRS batas administrasi belum terdefinisi.")

admin = admin[admin.geometry.notna() & ~admin.geometry.is_empty].copy()
admin = admin.to_crs(PROJECTED_CRS)

KELURAHAN_FIELD = None
KECAMATAN_FIELD = next(
    (c for c in ["WADMKC", "kecamatan", "KECAMATAN", "nama_kec", "NAMOBJ"] if c in admin.columns),
    None,
)
if KECAMATAN_FIELD is None:
    raise KeyError(f"Field kecamatan tidak ditemukan. Kolom tersedia: {list(admin.columns)}")

city_union = admin.geometry.union_all()
if city_union.is_empty:
    raise ValueError("Geometri gabungan batas Kota Bandung kosong.")

print("Jumlah poligon administrasi:", len(admin))
print("Field kecamatan:", KECAMATAN_FIELD)
print("CRS pemrosesan:", admin.crs)
display(admin[[KECAMATAN_FIELD, "geometry"]].head())

## 6. Memuat search grid 1 km atau membuatnya jika belum ada

Jika file `bandung_search_grid_1km.gpkg` sudah ada, notebook akan menggunakannya agar ID pilot tetap konsisten. Jika tidak ada, fishnet 1 km akan dibuat dari batas Kota Bandung.

ID yang sudah ada akan dipertahankan. Jika tidak ditemukan field ID, notebook membuat ID `BDG_0001`, `BDG_0002`, dan seterusnya berdasarkan urutan baris utara-ke-selatan lalu barat-ke-timur.

In [ ]:
def make_search_grid(boundary_gdf, cell_size=1000):
    """Buat fishnet meter, clip ke batas studi, dan beri ID deterministik."""
    geom = boundary_gdf.geometry.union_all()
    xmin, ymin, xmax, ymax = geom.bounds
    x0 = math.floor(xmin / cell_size) * cell_size
    y0 = math.floor(ymin / cell_size) * cell_size
    x1 = math.ceil(xmax / cell_size) * cell_size
    y1 = math.ceil(ymax / cell_size) * cell_size

    records = []
    y_top = y1
    row_no = 0
    y = y1 - cell_size
    while y < y1 + 1e-6 and y >= y0 - 1e-6:
        x = x0
        col_no = 0
        while x < x1 - 1e-6:
            raw_cell = box(x, y, x + cell_size, y + cell_size)
            clipped = raw_cell.intersection(geom)
            if not clipped.is_empty and clipped.area > 10:
                records.append({"row_no": row_no, "col_no": col_no, "geometry": clipped})
            x += cell_size
            col_no += 1
        y -= cell_size
        row_no += 1

    out = gpd.GeoDataFrame(records, geometry="geometry", crs=boundary_gdf.crs)
    out = out.sort_values(["row_no", "col_no"]).reset_index(drop=True)
    out["grid_id"] = [f"BDG_{i:04d}" for i in range(1, len(out) + 1)]
    out["cell_size_m"] = int(cell_size)
    return out

if SEARCH_GRID_PATH.exists():
    search_grid = gpd.read_file(SEARCH_GRID_PATH)
    if search_grid.crs is None:
        raise ValueError("CRS search grid belum terdefinisi.")
    search_grid = search_grid.to_crs(PROJECTED_CRS)
    id_field = next(
        (c for c in ["grid_id", "area_id", "cell_id", "id", "name", "Name"] if c in search_grid.columns),
        None,
    )
    if id_field is None:
        search_grid = search_grid.reset_index(drop=True)
        search_grid["grid_id"] = [f"BDG_{i:04d}" for i in range(1, len(search_grid) + 1)]
        print("Peringatan: ID asli tidak ditemukan; ID baru dibuat dari urutan fitur.")
    elif id_field != "grid_id":
        search_grid = search_grid.rename(columns={id_field: "grid_id"})
    search_grid["grid_id"] = search_grid["grid_id"].astype(str)
    print("Memakai search grid yang sudah ada.")
else:
    search_grid = make_search_grid(admin, cell_size=1000)
    SEARCH_GRID_PATH.parent.mkdir(parents=True, exist_ok=True)
    search_grid.to_file(SEARCH_GRID_PATH, layer="search_grid_1km", driver="GPKG")
    print("Search grid 1 km baru dibuat dan disimpan.")

search_grid = search_grid[search_grid.geometry.notna() & ~search_grid.geometry.is_empty].copy()
search_grid = search_grid.drop_duplicates("grid_id").reset_index(drop=True)

print("Jumlah search grid:", len(search_grid))
print("ID unik:", search_grid.grid_id.nunique())
print("CRS:", search_grid.crs)
display(search_grid[["grid_id", "geometry"]].head())

## 7. Validasi grid dan label kecamatan

Label kecamatan di sini adalah kecamatan yang memotong/beririsan dengan sel. Untuk memilih area berdasarkan kecamatan, gunakan ringkasan ini sebagai panduan; satu sel di batas dapat beririsan dengan lebih dari satu kecamatan.

In [ ]:
# Label kecamatan dominan berdasarkan luas irisan terbesar per sel
admin_for_overlay = admin[[KECAMATAN_FIELD, "geometry"]].copy()
overlay = gpd.overlay(
    search_grid[["grid_id", "geometry"]],
    admin_for_overlay,
    how="intersection",
    keep_geom_type=False,
)
overlay["intersection_area_m2"] = overlay.geometry.area
main_kec = (
    overlay.sort_values(["grid_id", "intersection_area_m2"], ascending=[True, False])
    .drop_duplicates("grid_id")[["grid_id", KECAMATAN_FIELD]]
    .rename(columns={KECAMATAN_FIELD: "kecamatan_dominan"})
)
search_grid = search_grid.merge(main_kec, on="grid_id", how="left")

print("Jumlah grid:", len(search_grid))
print("Distribusi sel menurut kecamatan dominan:")
display(search_grid["kecamatan_dominan"].value_counts(dropna=False).rename_axis("kecamatan").to_frame("jumlah_grid"))

## 8. Visualisasi area pencarian

Peta ini menampilkan seluruh search grid dan batas kecamatan. Sel yang akan diproses pada mode pilot/custom akan diberi warna berbeda pada cell pemilihan area berikutnya.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))
admin.boundary.plot(ax=ax, linewidth=0.6, color="black")
search_grid.boundary.plot(ax=ax, linewidth=0.45, color="steelblue")
ax.set_title(f"Search grid Mapillary Kota Bandung ({len(search_grid)} sel, 1 km)")
ax.set_axis_off()
plt.show()

## 9. Membuat bounding box WGS84 per search grid

Graph API menerima bbox dengan urutan `west,south,east,north` dalam koordinat geografis. Geometri grid terlebih dahulu diproyeksikan ke EPSG:4326, lalu bounds-nya dipakai sebagai bbox query.

In [ ]:
grid_wgs84 = search_grid.to_crs(GEOGRAPHIC_CRS).copy()

bbox_rows = []
for _, row in grid_wgs84.iterrows():
    west, south, east, north = row.geometry.bounds
    bbox_rows.append({
        "grid_id": str(row["grid_id"]),
        "kecamatan_dominan": row.get("kecamatan_dominan", None),
        "west": float(west),
        "south": float(south),
        "east": float(east),
        "north": float(north),
        "bbox": f"{west:.8f},{south:.8f},{east:.8f},{north:.8f}",
    })

bbox_table = pd.DataFrame(bbox_rows)
if bbox_table.empty:
    raise ValueError("Tidak ada bounding box yang berhasil dibuat.")

bbox_table.to_csv(METADATA_DIR / "search_grid_bboxes.csv", index=False)
print("Bbox tersimpan:", METADATA_DIR / "search_grid_bboxes.csv")
display(bbox_table.head())

## 10. Memilih area yang akan dikumpulkan

Default adalah lima grid pilot yang sudah digunakan sebelumnya. Jika salah satu ID pilot tidak ditemukan pada file grid, notebook akan menampilkan peringatan dan mengambil lima sel pertama sebagai fallback **hanya untuk uji coba**. Periksa peta dan daftar ID sebelum menjalankan API.

In [ ]:
available_ids = set(search_grid["grid_id"].astype(str))

if COLLECTION_MODE == "pilot":
    found_pilot = [x for x in PILOT_AREA_IDS if x in available_ids]
    missing_pilot = [x for x in PILOT_AREA_IDS if x not in available_ids]
    if missing_pilot:
        print("Peringatan, ID pilot tidak ditemukan:", missing_pilot)
    selected_ids = found_pilot if found_pilot else search_grid.head(5)["grid_id"].astype(str).tolist()
elif COLLECTION_MODE == "custom":
    if not CUSTOM_AREA_IDS:
        raise ValueError("CUSTOM_AREA_IDS masih kosong.")
    missing = [x for x in CUSTOM_AREA_IDS if x not in available_ids]
    if missing:
        raise KeyError(f"ID custom tidak ditemukan: {missing}")
    selected_ids = CUSTOM_AREA_IDS
elif COLLECTION_MODE == "all":
    if not ALLOW_FULL_COLLECTION:
        raise RuntimeError("Mode all dipilih, tetapi ALLOW_FULL_COLLECTION=False. Ubah secara sadar setelah pilot tervalidasi.")
    selected_ids = search_grid["grid_id"].astype(str).tolist()
else:
    raise ValueError("COLLECTION_MODE harus 'pilot', 'custom', atau 'all'.")

area_selection = bbox_table[bbox_table.grid_id.isin(selected_ids)].copy()
area_selection = area_selection.sort_values("grid_id").reset_index(drop=True)

print("Mode:", COLLECTION_MODE)
print("Area terpilih:", len(area_selection))
display(area_selection[["grid_id", "kecamatan_dominan", "bbox"]])

### Preview area terpilih

Periksa bahwa sel yang dipilih sesuai dengan cakupan yang diinginkan sebelum mengaktifkan `RUN_COLLECTION`.

In [ ]:
selected_grid = search_grid[search_grid.grid_id.astype(str).isin(selected_ids)]
fig, ax = plt.subplots(figsize=(10, 10))
admin.boundary.plot(ax=ax, linewidth=0.6, color="black")
search_grid.boundary.plot(ax=ax, linewidth=0.35, color="lightgray")
selected_grid.boundary.plot(ax=ax, linewidth=1.2, color="darkorange")
ax.set_title(f"Area pengumpulan terpilih: {len(selected_grid)} search grid")
ax.set_axis_off()
plt.show()

## 11. Fungsi API, retry, dan recursive bbox split

Setiap bbox akan di-query ke endpoint `GET https://graph.mapillary.com/images`. Field yang diminta meliputi ID, geometri, waktu pengambilan, sequence, arah kamera, URL thumbnail, creator, dan informasi kamera/resolusi.

Jika jumlah hasil mencapai `SPLIT_THRESHOLD`, query dipecah menjadi empat kuadran. Jika API gagal dengan status server (misalnya HTTP 500) setelah retry, fungsi juga mencoba membagi bbox. Kesalahan autentikasi atau parameter (401/403/400) tidak disamarkan sebagai masalah bbox.

Dokumentasi endpoint dan field: [Mapillary API entities](https://mapillary.github.io/mapillary-python-sdk/docs/mapillary.config.api/mapillary.config.api.entities/).

In [ ]:
API_STATS = {"requests": 0, "splits": 0, "retries": 0}

class MapillaryRequestError(RuntimeError):
    pass


def request_json(url, params=None):
    """GET JSON dengan retry terbatas dan tanpa mencetak token/URL sensitif."""
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            API_STATS["requests"] += 1
            response = SESSION.get(url, params=params, timeout=REQUEST_TIMEOUT)
            if response.status_code == 429:
                last_error = requests.HTTPError("HTTP 429: rate limit Mapillary")
                wait = response.headers.get("Retry-After")
                wait_seconds = float(wait) if wait and str(wait).replace(".", "", 1).isdigit() else min(2 ** attempt, 30)
                API_STATS["retries"] += 1
                time.sleep(wait_seconds)
                continue
            if response.status_code in (500, 502, 503, 504):
                last_error = requests.HTTPError(f"HTTP {response.status_code} dari Mapillary")
                API_STATS["retries"] += 1
                time.sleep(min(2 ** attempt, 20))
                continue
            response.raise_for_status()
            payload = response.json()
            time.sleep(REQUEST_PAUSE)
            return payload
        except (requests.Timeout, requests.ConnectionError, requests.HTTPError, ValueError) as exc:
            last_error = exc
            status = getattr(getattr(exc, "response", None), "status_code", None)
            # Kesalahan client/auth tidak perlu diulang.
            if status in (400, 401, 403, 404):
                raise MapillaryRequestError(f"Request ditolak Mapillary (HTTP {status}). Periksa token/parameter/izin.") from exc
            if attempt < MAX_RETRIES - 1:
                API_STATS["retries"] += 1
                time.sleep(min(2 ** attempt, 20))
    raise MapillaryRequestError(f"Request Mapillary gagal setelah {MAX_RETRIES} percobaan: {type(last_error).__name__}: {last_error}")


def split_bbox(bbox):
    west, south, east, north = bbox
    mid_lon = (west + east) / 2.0
    mid_lat = (south + north) / 2.0
    return [
        (west, south, mid_lon, mid_lat),
        (mid_lon, south, east, mid_lat),
        (west, mid_lat, mid_lon, north),
        (mid_lon, mid_lat, east, north),
    ]


def can_split_bbox(bbox, depth):
    west, south, east, north = bbox
    return (
        depth < MAX_SPLIT_DEPTH
        and (east - west) > (2 * MIN_BBOX_DEG)
        and (north - south) > (2 * MIN_BBOX_DEG)
    )


def bbox_to_string(bbox):
    return ",".join(f"{v:.8f}" for v in bbox)


def collect_bbox_recursive(bbox, depth=0):
    """Ambil seluruh gambar di bbox; subdivide bila bbox padat atau server gagal."""
    params = {
        "bbox": bbox_to_string(bbox),
        "fields": ",".join(API_FIELDS),
        "limit": API_LIMIT,
    }
    try:
        payload = request_json(MAPILLARY_ENDPOINT, params=params)
    except MapillaryRequestError as exc:
        if can_split_bbox(bbox, depth) and not any(code in str(exc) for code in ["HTTP 400", "HTTP 401", "HTTP 403", "HTTP 404"]):
            API_STATS["splits"] += 1
            child_results = []
            for child_bbox in split_bbox(bbox):
                child_results.extend(collect_bbox_recursive(child_bbox, depth + 1))
            return child_results
        raise

    records = payload.get("data", []) or []
    next_url = (payload.get("paging") or {}).get("next")

    # Respon padat: query ulang lewat empat sub-bbox agar hasil tidak terpotong.
    if len(records) >= SPLIT_THRESHOLD and can_split_bbox(bbox, depth):
        API_STATS["splits"] += 1
        child_results = []
        for child_bbox in split_bbox(bbox):
            child_results.extend(collect_bbox_recursive(child_bbox, depth + 1))
        return child_results

    # Jika bbox tidak padat, lanjutkan pagination sampai selesai.
    all_records = list(records)
    while next_url:
        next_payload = request_json(next_url, params=None)
        all_records.extend(next_payload.get("data", []) or [])
        next_url = (next_payload.get("paging") or {}).get("next")

        if len(all_records) >= SPLIT_THRESHOLD and can_split_bbox(bbox, depth):
            API_STATS["splits"] += 1
            child_results = []
            for child_bbox in split_bbox(bbox):
                child_results.extend(collect_bbox_recursive(child_bbox, depth + 1))
            return child_results

    return all_records

print("Fungsi API siap.")

## 12. Normalisasi metadata hasil API

Respons API disimpan ke skema kolom yang konsisten dengan alur proyek. Metadata `license` tidak diisi dengan tebakan; lisensi dan ketentuan penggunaan perlu diperiksa sesuai ketentuan Mapillary yang berlaku. Field creator dipertahankan untuk kebutuhan atribusi.

In [ ]:
MASTER_COLUMNS = [
    "image_id", "longitude", "latitude", "captured_at_ms", "captured_at_wib", "date_token",
    "sequence_id", "compass_angle", "creator_username", "creator_id", "camera_type", "width", "height",
    "thumb_1024_url", "thumb_2048_url", "collection_area", "kecamatan", "image_page_url", "license",
    "visual_category", "selected", "download_status", "filename", "sha256", "collection_areas",
]


def nested_id(value):
    if isinstance(value, dict):
        return value.get("id")
    return value


def flatten_api_record(record, area_id):
    geom = record.get("computed_geometry") or record.get("geometry") or {}
    coords = geom.get("coordinates", []) if isinstance(geom, dict) else []
    longitude = coords[0] if len(coords) >= 2 else np.nan
    latitude = coords[1] if len(coords) >= 2 else np.nan

    creator = record.get("creator") or {}
    if not isinstance(creator, dict):
        creator = {}
    sequence = record.get("sequence")
    sequence_id = nested_id(sequence)
    captured_at = record.get("captured_at")
    creator_id = creator.get("id")
    creator_username = creator.get("username")
    image_id = record.get("id")

    return {
        "image_id": str(image_id) if image_id is not None else None,
        "longitude": longitude,
        "latitude": latitude,
        "captured_at_ms": captured_at,
        "captured_at_wib": None,
        "date_token": None,
        "sequence_id": str(sequence_id) if sequence_id is not None else None,
        "compass_angle": record.get("compass_angle"),
        "creator_username": creator_username,
        "creator_id": str(creator_id) if creator_id is not None else None,
        "camera_type": record.get("camera_type"),
        "width": record.get("width"),
        "height": record.get("height"),
        "thumb_1024_url": record.get("thumb_1024_url"),
        "thumb_2048_url": record.get("thumb_2048_url"),
        "collection_area": str(area_id),
        "kecamatan": None,
        "image_page_url": f"https://www.mapillary.com/app/?pKey={image_id}" if image_id is not None else None,
        "license": None,
        "visual_category": None,
        "selected": None,
        "download_status": None,
        "filename": None,
        "sha256": None,
        "collection_areas": str(area_id),
    }


def records_to_dataframe(records, area_id):
    rows = [flatten_api_record(item, area_id) for item in records]
    df = pd.DataFrame(rows, columns=MASTER_COLUMNS)
    if not df.empty:
        df = df[df["image_id"].notna()].copy()
        df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
        df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    return df

print("Skema metadata siap:", len(MASTER_COLUMNS), "kolom")

## 13. Menjalankan pengumpulan per area dan menyimpan checkpoint

Setiap search grid disimpan ke CSV tersendiri di `METADATA/by_grid/`. Jika sesi Colab terputus, jalankan kembali notebook dengan `RECOLLECT_COMPLETED = False`; file area yang sudah ada akan dilewati.

Sebelum collection berjalan, pastikan:
- Token berhasil dimasukkan.
- `area_selection` menampilkan grid yang benar.
- `RUN_COLLECTION = True`.
- Mode `all` hanya digunakan setelah pilot berhasil.

In [ ]:
def save_collection_log(rows, path):
    pd.DataFrame(rows).to_csv(path, index=False)

collection_log_path = LOG_DIR / "collection_log.csv"
existing_log = pd.read_csv(collection_log_path) if collection_log_path.exists() else pd.DataFrame()

if RUN_COLLECTION:
    if COLLECTION_MODE == "all" and not ALLOW_FULL_COLLECTION:
        raise RuntimeError("Full collection diblokir: set ALLOW_FULL_COLLECTION=True setelah meninjau area.")

    log_rows = existing_log.to_dict("records") if not existing_log.empty else []
    log_by_id = {str(x.get("grid_id")): x for x in log_rows if x.get("grid_id") is not None}

    for area in tqdm(area_selection.to_dict("records"), desc="Collect Mapillary per grid"):
        grid_id = str(area["grid_id"])
        area_csv = PER_AREA_DIR / f"{grid_id}.csv"

        if area_csv.exists() and not RECOLLECT_COMPLETED:
            print(f"Skip {grid_id}: checkpoint sudah tersedia")
            continue

        bbox = (float(area["west"]), float(area["south"]), float(area["east"]), float(area["north"]))
        started = datetime.now(timezone.utc).isoformat()
        req_before = API_STATS["requests"]
        split_before = API_STATS["splits"]

        try:
            api_records = collect_bbox_recursive(bbox, depth=0)
            area_df = records_to_dataframe(api_records, grid_id)
            # Deduplikasi lokal dalam sel sebelum checkpoint.
            if not area_df.empty:
                area_df = area_df.drop_duplicates("image_id", keep="first")
            area_df.to_csv(area_csv, index=False)

            log_by_id[grid_id] = {
                "grid_id": grid_id,
                "kecamatan_dominan": area.get("kecamatan_dominan"),
                "status": "completed",
                "image_count_unique_in_area": int(len(area_df)),
                "started_utc": started,
                "finished_utc": datetime.now(timezone.utc).isoformat(),
                "api_requests": API_STATS["requests"] - req_before,
                "recursive_splits": API_STATS["splits"] - split_before,
                "error": None,
            }
            print(f"{grid_id}: {len(area_df):,} image unik disimpan")

        except Exception as exc:
            log_by_id[grid_id] = {
                "grid_id": grid_id,
                "kecamatan_dominan": area.get("kecamatan_dominan"),
                "status": "failed",
                "image_count_unique_in_area": 0,
                "started_utc": started,
                "finished_utc": datetime.now(timezone.utc).isoformat(),
                "api_requests": API_STATS["requests"] - req_before,
                "recursive_splits": API_STATS["splits"] - split_before,
                "error": f"{type(exc).__name__}: {exc}",
            }
            print(f"GAGAL {grid_id}: {type(exc).__name__}: {exc}")

        save_collection_log(list(log_by_id.values()), collection_log_path)

    print("Pengumpulan area selesai/berhenti.")
    print("Log:", collection_log_path)
    print("API stats:", API_STATS)
else:
    print("RUN_COLLECTION=False. Belum ada request pencarian gambar yang dijalankan.")
    print("Set RUN_COLLECTION=True setelah memeriksa token dan area_selection.")

## 14. Ringkasan status area dan checkpoint

Cell ini dapat dijalankan meskipun pengumpulan baru dilakukan sebagian.

In [ ]:
if collection_log_path.exists():
    collection_log = pd.read_csv(collection_log_path)
    display(collection_log.sort_values("grid_id"))
    print(collection_log["status"].value_counts(dropna=False))
else:
    print("Belum ada collection_log.csv.")

checkpoint_files = sorted(PER_AREA_DIR.glob("*.csv"))
print("Jumlah file checkpoint:", len(checkpoint_files))
print("Folder:", PER_AREA_DIR)

## 15. Menggabungkan checkpoint menjadi master dataset

Semua checkpoint CSV pada `by_grid/` digabungkan. Duplikasi antar-area dihapus berdasarkan `image_id`, sedangkan daftar area tempat gambar ditemukan tetap disimpan pada `collection_areas`. Ini penting karena bbox grid yang bersebelahan dapat saling overlap di batasnya.

In [ ]:
def read_checkpoint_csv(path):
    try:
        frame = pd.read_csv(path)
        if "image_id" not in frame.columns:
            return pd.DataFrame(columns=MASTER_COLUMNS)
        if "collection_area" not in frame.columns:
            frame["collection_area"] = path.stem
        return frame
    except pd.errors.EmptyDataError:
        return pd.DataFrame(columns=MASTER_COLUMNS)

frames = [read_checkpoint_csv(p) for p in checkpoint_files]
frames = [f for f in frames if not f.empty]

if not frames:
    print("Belum ada checkpoint berisi gambar. Jalankan collection terlebih dahulu.")
    candidates_master = pd.DataFrame(columns=MASTER_COLUMNS)
else:
    collected = pd.concat(frames, ignore_index=True, sort=False)
    collected["image_id"] = collected["image_id"].astype(str)
    collected["collection_area"] = collected["collection_area"].fillna("UNKNOWN").astype(str)

    area_lookup = (
        collected.groupby("image_id")["collection_area"]
        .apply(lambda x: ",".join(sorted(set(v for v in x.astype(str) if v and v != "nan"))))
    )
    candidates_master = collected.drop_duplicates("image_id", keep="first").copy()
    candidates_master["collection_areas"] = candidates_master["image_id"].map(area_lookup)
    candidates_master = candidates_master.reindex(columns=MASTER_COLUMNS)

    print("Record dari seluruh checkpoint:", len(collected))
    print("Image ID unik:", len(candidates_master))
    print("Duplikasi antar-area:", len(collected) - len(candidates_master))
    display(candidates_master.head())

## 16. Membersihkan koordinat, timestamp, dan memastikan titik berada di Kota Bandung

Bbox API berbentuk persegi panjang, sehingga pada batas sel/kota API mungkin mengembalikan titik di luar poligon studi. Titik tersebut dikeluarkan menggunakan `covers()` terhadap union batas kecamatan.

In [ ]:
if not candidates_master.empty:
    candidates_master["longitude"] = pd.to_numeric(candidates_master["longitude"], errors="coerce")
    candidates_master["latitude"] = pd.to_numeric(candidates_master["latitude"], errors="coerce")
    candidates_master["captured_at_ms"] = pd.to_numeric(candidates_master["captured_at_ms"], errors="coerce")

    candidates_master = candidates_master.dropna(subset=["longitude", "latitude"]).copy()
    valid_coord = (
        candidates_master["longitude"].between(-180, 180)
        & candidates_master["latitude"].between(-90, 90)
    )
    candidates_master = candidates_master[valid_coord].copy()

    # API captured_at menggunakan epoch milliseconds.
    captured_utc = pd.to_datetime(candidates_master["captured_at_ms"], unit="ms", utc=True, errors="coerce")
    captured_wib = captured_utc.dt.tz_convert("Asia/Jakarta")
    candidates_master["captured_at_wib"] = captured_wib.astype("string")
    candidates_master["date_token"] = captured_wib.dt.strftime("%Y%m%d")

    points_utm = gpd.GeoDataFrame(
        candidates_master.copy(),
        geometry=gpd.points_from_xy(candidates_master.longitude, candidates_master.latitude),
        crs=GEOGRAPHIC_CRS,
    ).to_crs(PROJECTED_CRS)

    # covers termasuk titik tepat di batas poligon.
    inside = points_utm.geometry.apply(city_union.covers)
    before = len(points_utm)
    points_utm = points_utm.loc[inside].copy()
    candidates_master = pd.DataFrame(points_utm.drop(columns="geometry"))

    print("Titik valid sebelum filter batas:", before)
    print("Titik di dalam/tepat pada batas studi:", len(candidates_master))
else:
    print("Master kosong; tahap filter koordinat dilewati.")

## 17. Menambahkan label kecamatan dari lokasi kamera

Titik Mapillary menunjukkan lokasi kamera. Label kecamatan diberikan berdasarkan lokasi kamera, bukan klaim bahwa titik tersebut adalah koordinat persis permukaan trotoar yang terlihat.

In [ ]:
if not candidates_master.empty:
    points_wgs = gpd.GeoDataFrame(
        candidates_master.copy(),
        geometry=gpd.points_from_xy(candidates_master.longitude, candidates_master.latitude),
        crs=GEOGRAPHIC_CRS,
    ).to_crs(PROJECTED_CRS)

    admin_join = admin[[KECAMATAN_FIELD, "geometry"]].copy()
    joined = gpd.sjoin(points_wgs, admin_join, how="left", predicate="within")

    # Fallback intersects untuk titik persis pada garis batas kecamatan.
    missing_kec = joined[KECAMATAN_FIELD].isna()
    if missing_kec.any():
        fallback = gpd.sjoin(
            points_wgs.loc[missing_kec, ["image_id", "geometry"]],
            admin_join,
            how="left",
            predicate="intersects",
        )
        fallback = fallback.drop_duplicates("image_id").set_index("image_id")[KECAMATAN_FIELD]
        joined.loc[missing_kec, KECAMATAN_FIELD] = joined.loc[missing_kec, "image_id"].map(fallback)

    kec_lookup = joined.drop_duplicates("image_id").set_index("image_id")[KECAMATAN_FIELD]
    candidates_master["kecamatan"] = candidates_master["image_id"].map(kec_lookup)
    candidates_master = candidates_master.rename(columns={KECAMATAN_FIELD: "kecamatan"}) if KECAMATAN_FIELD in candidates_master.columns else candidates_master

    display(candidates_master["kecamatan"].value_counts(dropna=False).rename_axis("kecamatan_kamera").to_frame("jumlah_gambar"))
else:
    print("Master kosong; spatial join kecamatan dilewati.")

## 18. Menyimpan `candidates_master.csv`

File master menjadi basis pengelolaan seluruh image ID unik sebelum spatial sampling. Metadata mentah hasil API tetap tersedia di file per-area sebagai checkpoint.

In [ ]:
MASTER_CSV = METADATA_DIR / "candidates_master.csv"
if not candidates_master.empty:
    candidates_master = candidates_master.reindex(columns=MASTER_COLUMNS)
    candidates_master.to_csv(MASTER_CSV, index=False)
    print("Master tersimpan:", MASTER_CSV)
    print("Jumlah image unik:", len(candidates_master))
else:
    print("Master belum ditulis karena belum ada data.")

## 19. Spatial sampling minimum 10 meter dalam sequence yang sama

Sampling memakai greedy distance filter:
- Foto dikelompokkan berdasarkan `sequence_id`.
- Dalam setiap sequence, foto diurutkan berdasarkan waktu pengambilan.
- Foto dipertahankan jika jaraknya minimal 10 m dari foto yang sudah dipertahankan pada sequence tersebut.
- Foto tanpa `sequence_id` diperlakukan sebagai sequence tersendiri.

**Batasan:** sampling ini tidak menghapus foto mirip yang berasal dari sequence berbeda. Deduplikasi lintas sequence perlu pemeriksaan tambahan.

In [ ]:
def haversine_m(lon1, lat1, lon2, lat2):
    """Jarak great-circle dalam meter; menerima scalar/array untuk titik 2."""
    earth_r = 6371008.8
    lon1 = np.radians(float(lon1))
    lat1 = np.radians(float(lat1))
    lon2 = np.radians(np.asarray(lon2, dtype=float))
    lat2 = np.radians(np.asarray(lat2, dtype=float))
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * earth_r * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def spatial_sample_by_sequence(df, min_distance_m=10.0):
    if df.empty:
        return df.copy()
    work = df.copy()
    work["_seq_group"] = work["sequence_id"].fillna("__NO_SEQUENCE__" + work["image_id"].astype(str))
    work["_sort_time"] = pd.to_numeric(work["captured_at_ms"], errors="coerce")
    work = work.sort_values(["_seq_group", "_sort_time", "image_id"], na_position="last")

    keep_indices = []
    for _, group in tqdm(work.groupby("_seq_group", sort=False), desc="Sampling by sequence"):
        chosen_lon = []
        chosen_lat = []
        for idx, row in group.iterrows():
            lon = float(row["longitude"])
            lat = float(row["latitude"])
            if not chosen_lon:
                keep_indices.append(idx)
                chosen_lon.append(lon)
                chosen_lat.append(lat)
                continue
            distances = haversine_m(lon, lat, chosen_lon, chosen_lat)
            if np.all(distances >= min_distance_m):
                keep_indices.append(idx)
                chosen_lon.append(lon)
                chosen_lat.append(lat)

    result = work.loc[keep_indices].drop(columns=["_seq_group", "_sort_time"], errors="ignore").copy()
    return result.sort_values(["collection_area", "sequence_id", "captured_at_ms"], na_position="last").reset_index(drop=True)

if not candidates_master.empty:
    sampled_candidates = spatial_sample_by_sequence(candidates_master, MIN_DISTANCE_METERS)
    sampled_candidates["selected"] = True
    print("Sebelum sampling:", len(candidates_master))
    print("Setelah sampling:", len(sampled_candidates))
    print("Dihilangkan oleh sampling:", len(candidates_master) - len(sampled_candidates))
else:
    sampled_candidates = pd.DataFrame(columns=MASTER_COLUMNS)
    print("Master kosong; sampling dilewati.")

## 20. Menyimpan kandidat 10 m untuk tahap screening/download

Output ini mempertahankan struktur kolom yang digunakan notebook screening sebelumnya.

In [ ]:
SAMPLED_CSV = METADATA_DIR / "review_candidates_10m_full.csv"
if not sampled_candidates.empty:
    sampled_candidates = sampled_candidates.reindex(columns=MASTER_COLUMNS)
    sampled_candidates.to_csv(SAMPLED_CSV, index=False)
    print("Kandidat 10 m tersimpan:", SAMPLED_CSV)
else:
    print("Belum ada kandidat hasil sampling untuk disimpan.")

## 21. Ringkasan per area dan per kecamatan

In [ ]:
if checkpoint_files:
    area_summary_rows = []
    for p in checkpoint_files:
        area_df = read_checkpoint_csv(p)
        area_summary_rows.append({
            "grid_id": p.stem,
            "image_count_unique_in_area": int(area_df["image_id"].nunique()) if not area_df.empty else 0,
            "sequence_count": int(area_df["sequence_id"].nunique()) if not area_df.empty and "sequence_id" in area_df else 0,
        })
    area_summary = pd.DataFrame(area_summary_rows)
    area_summary = area_summary.merge(
        search_grid[["grid_id", "kecamatan_dominan"]], on="grid_id", how="left"
    )
    area_summary.to_csv(METADATA_DIR / "collection_area_summary.csv", index=False)
    display(area_summary.sort_values("image_count_unique_in_area", ascending=False))
else:
    area_summary = pd.DataFrame()
    print("Belum ada checkpoint area.")

if not candidates_master.empty:
    print("\nDistribusi master per kecamatan kamera:")
    display(candidates_master["kecamatan"].value_counts(dropna=False).to_frame("unique_images"))
    print("\nSequence unik:", candidates_master["sequence_id"].nunique(dropna=True))

## 22. Ekspor titik Mapillary ke GeoPackage

CSV tetap menjadi master tabel. GeoPackage dibuat untuk inspeksi spasial di QGIS.

In [ ]:
if not sampled_candidates.empty:
    sample_points = gpd.GeoDataFrame(
        sampled_candidates.copy(),
        geometry=gpd.points_from_xy(sampled_candidates.longitude, sampled_candidates.latitude),
        crs=GEOGRAPHIC_CRS,
    ).to_crs(PROJECTED_CRS)
    sample_gpkg = METADATA_DIR / "mapillary_candidates_10m.gpkg"
    if sample_gpkg.exists():
        sample_gpkg.unlink()
    sample_points.to_file(sample_gpkg, layer="mapillary_candidates_10m", driver="GPKG")
    print("GeoPackage tersimpan:", sample_gpkg)
else:
    print("Tidak ada titik untuk diekspor ke GeoPackage.")

## 23. Pemeriksaan kualitas dataset

Pemeriksaan ini bukan screening visual. Tujuannya memastikan metadata cukup lengkap untuk tahapan selanjutnya.

In [ ]:
if not candidates_master.empty:
    quality = pd.DataFrame({
        "field": ["image_id", "longitude", "latitude", "captured_at_ms", "sequence_id", "compass_angle", "thumb_1024_url", "creator_username", "camera_type", "width", "height", "kecamatan"],
        "missing_count": [int(candidates_master[c].isna().sum()) if c in candidates_master else len(candidates_master) for c in ["image_id", "longitude", "latitude", "captured_at_ms", "sequence_id", "compass_angle", "thumb_1024_url", "creator_username", "camera_type", "width", "height", "kecamatan"]],
    })
    quality["missing_pct"] = (quality["missing_count"] / len(candidates_master) * 100).round(2)
    display(quality)
    print("Duplicate image_id:", int(candidates_master.image_id.duplicated().sum()))
    print("Koordinat valid:", int(candidates_master[["longitude", "latitude"]].notna().all(axis=1).sum()))
    print("Waktu capture valid:", int(candidates_master["captured_at_ms"].notna().sum()))
else:
    print("Belum ada data untuk quality check.")

## 24. Inventaris output

Output utama yang dihasilkan:

| File | Fungsi |
|---|---|
| `search_grid_bboxes.csv` | Bbox WGS84 per search grid |
| `by_grid/BDG_xxxx.csv` | Checkpoint API per area |
| `logs/collection_log.csv` | Status, jumlah hasil, request, split, dan error per area |
| `candidates_master.csv` | Semua image ID unik sebelum sampling |
| `review_candidates_10m_full.csv` | Kandidat setelah sampling 10 m dalam sequence |
| `collection_area_summary.csv` | Ringkasan hasil per grid |
| `mapillary_candidates_10m.gpkg` | Titik kandidat untuk dibuka di QGIS |

Notebook berikutnya dapat memakai `review_candidates_10m_full.csv` sebagai input download dan screening.

In [ ]:
expected_outputs = [
    METADATA_DIR / "search_grid_bboxes.csv",
    collection_log_path,
    METADATA_DIR / "candidates_master.csv",
    METADATA_DIR / "review_candidates_10m_full.csv",
    METADATA_DIR / "collection_area_summary.csv",
    METADATA_DIR / "mapillary_candidates_10m.gpkg",
]

for p in expected_outputs:
    print(("[ADA] " if p.exists() else "[BELUM ADA] ") + str(p))

print("Checkpoint per area:", len(list(PER_AREA_DIR.glob("*.csv"))))

## 25. Melanjutkan collection setelah terputus

1. Buka kembali notebook dan mount Google Drive.
2. Gunakan `COLLECTION_MODE` dan daftar area yang sama.
3. Masukkan token Mapillary kembali melalui `getpass`.
4. Biarkan `RECOLLECT_COMPLETED = False`.
5. Ubah `RUN_COLLECTION = True` dan jalankan cell pengumpulan.
6. File checkpoint yang sudah ada akan dilewati; area yang gagal dapat dihapus file checkpoint-nya (jika ada) dan dijalankan ulang.

**Jangan menghapus `by_grid/` atau master CSV sebelum memastikan salinan backup tersedia.** Jika ingin mengulang semua area dari nol, ubah nama folder output lama sebagai arsip terlebih dahulu.

## 26. Batasan dan keputusan metodologis

- **Lokasi kamera bukan lokasi pasti permukaan trotoar.** Compass angle dan pemeriksaan visual masih diperlukan untuk mengaitkan citra ke segmen pedestrian.
- **Bounding box dapat overlap.** Deduplikasi berdasarkan `image_id` dilakukan setelah penggabungan; daftar grid asal dipertahankan di `collection_areas`.
- **Spatial sampling 10 m hanya dalam sequence yang sama.** Citra mirip lintas sequence masih mungkin tersisa.
- **Metadata API bukan ground truth.** Kategori visual dan label trotoar belum ditetapkan pada notebook ini.
- **URL thumbnail dapat berubah/expired.** Refresh metadata untuk image ID terpilih jika URL tidak lagi dapat digunakan.
- **Atribusi dan izin penggunaan harus diperiksa.** Simpan image ID, creator, URL halaman Mapillary, dan dokumentasikan ketentuan lisensi/penggunaan yang berlaku. Kolom `license` sengaja tidak diisi otomatis.
- **Dataset saat ini bersifat pilot.** Lima grid pilot tidak mewakili seluruh Kota Bandung; jangan menggeneralisasi hasilnya ke seluruh kota sebelum cakupan diperluas.

Referensi teknis:
- [Mapillary API – accessing imagery and data](https://help.mapillary.com/hc/en-us/articles/360010234680-Accessing-imagery-and-data-through-the-Mapillary-API)
- [Mapillary API entities and image fields](https://mapillary.github.io/mapillary-python-sdk/docs/mapillary.config.api/mapillary.config.api.entities/)
- [Mapillary Python SDK – authentication](https://mapillary.github.io/mapillary-python-sdk/docs/mapillary.models/mapillary.models.client/)
- [Mapillary developer portal](https://www.mapillary.com/developer)

## 27. Tahap berikutnya

Setelah metadata Mapillary untuk seluruh area yang dibutuhkan berhasil dikumpulkan:

**Notebook 02 / Download & Screening**
1. Baca `review_candidates_10m_full.csv`.
2. Refresh detail/thumbnail URL bila diperlukan.
3. Download citra dengan pencatatan status, nama file, ukuran, dan SHA-256.
4. Terapkan quality screening dan pemeriksaan visual.
5. Siapkan citra terpilih untuk perbandingan model segmentasi.

Tahap segmentasi, validasi ground truth, dan PCD tetap dipisahkan dari proses koleksi API.